# 02 -- Phase 1: Calibration (ISR)

Instrument Signature Removal: build master bias/dark/flat, subtract/divide
them from each science frame, reject cosmic rays, convert ADU to electrons,
and **seed the error budget** into the ERR plane while flagging DQ.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

## Run Phase 1
Reads raw frames from `RAW_DIR`, writes `calibrated_*.fits` (SCI/ERR/DQ) into `PHASE1_DIR`.

`RAW_DIR` is searched **recursively**, so the night tree the acquisition
software writes -- `<date>/BIAS|DARK|FLAT|LIGHT/<target>/` -- is handed over
as-is. The folder names are not what sorts the frames: Phase 1 reads `IMAGETYP`
from each header, exactly as it does for a flat directory of simulated frames.

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase1_calibration import run as run_p1
cfg = load_config()
run_p1(RAW_DIR, PHASE1_DIR, config=cfg)

## Inspect a calibrated frame
Note the ERR plane is now populated and DQ carries saturation / CR / bad-pixel flags.

In [ ]:
import glob, os, numpy as np
from cassa_photometry.fits_utils import read_mef, DQ_FLAG_NAMES
out = sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits')))
print(len(out), 'calibrated frames')
sci, err, dq, hdr = read_mef(out[0])
print('BUNIT:', hdr.get('BUNIT'))
print('median SCI (e-):', float(np.nanmedian(sci)))
print('median ERR (e-):', float(np.nanmedian(err)))
for flag, name in DQ_FLAG_NAMES.items():
    print(f'  {name:>11}:', int(np.count_nonzero(dq & flag)), 'px')

### Exercise 1 -- what did calibration actually change?

Load the raw counterpart of the calibrated frame above (follow the
`RAWFILE`/`RAWDIR` cards), convert it to electrons with the gain Phase 1 used,
and take a sigma-clipped background of each.

| Find | Expected |
|---|---|
| instrumental pedestal removed (bias + dark) | `308` e- |
| real sky left behind | `103` e- |

_Expected values come from the reference reduction of the shipped night (NGC 7331, 2023-08-23)._

In [ ]:
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from cassa_photometry.instruments import get_profile

# Fill in the blanks marked TODO. Everything else is scaffolding.

# Phase 1 stamped where the raw frame lives, so follow that rather than
# rebuilding a path -- which subdirectory it came from is not ours to guess.
raw_path = os.path.join(hdr['RAWDIR'], hdr['RAWFILE'])
raw_header = fits.getheader(raw_path)
raw_adu = fits.getdata(raw_path).astype(np.float64)
instrument = get_profile(cfg.instrument, config=cfg)

gain = FILL_IN        # TODO 1: e-/ADU as Phase 1 resolved it (profile first, then cfg fallback)
raw_e = FILL_IN       # TODO 2: the raw frame converted to electrons

# Sigma-clipped, because a plain median is pulled up by every star in the field.
_, raw_sky, _ = sigma_clipped_stats(raw_e, sigma=3.0)
_, cal_sky, _ = sigma_clipped_stats(sci, sigma=3.0, mask=dq != 0)

pedestal = FILL_IN    # TODO 3: the instrumental level calibration took out

print(f"gain {gain} e-/ADU\n")
print(f"raw sky                        : {raw_sky:8.2f} e-")
print(f"calibrated sky                 : {cal_sky:8.2f} e-\n")
print(f"instrumental pedestal removed  : {pedestal:8.2f} e-")
print(f"real sky left behind           : {cal_sky:8.2f} e-")

### Exercise 2 -- is the ERR plane just Poisson + read noise?

Compare the `ERR` plane against $\sqrt{S + \mathrm{RN}^2}$, the error you would
predict if the science frame were the only noise source.

| Find | Expected |
|---|---|
| ratio, actual `ERR` / predicted | `1.05` |
| what the excess is | `4.8` e- — the master calibrations' own uncertainty |

In [ ]:
# `instrument` comes from Exercise 1, resolved from the config Phase 1 ran with.
# Fill in the blanks marked TODO. Everything else is scaffolding.
read_noise = instrument.get_read_noise(hdr) or cfg.phase1.fallback_read_noise
good = dq == 0

predicted = FILL_IN   # TODO 1: sqrt(S + RN^2) per pixel, S in electrons (clip S at 0 first)

actual_med = float(np.median(err[good]))
predicted_med = float(np.median(predicted[good]))
ratio = actual_med / predicted_med

extra = FILL_IN       # TODO 2: the term that, added in quadrature, closes the gap

print(f"read noise {read_noise} e-\n")
print(f"median ERR, actual    : {actual_med:6.2f} e-")
print(f"median ERR, predicted : {predicted_med:6.2f} e-")
print(f"ratio                 : {ratio:6.2f}")
print(f"extra term            : {extra:6.2f} e-  (in quadrature)\n")

print("The extra term is the calibration frames' own uncertainty: subtracting the")
print("master bias and dark, and dividing by the flat, each fold theirs into the")
print("budget, and ccdproc propagates all of it. How big it is depends on how many")
if ratio > 1.02:
    print("frames each master was built from -- here it is large enough to matter.")
else:
    print("frames each master was built from -- here they were built from enough that")
    print("it is negligible beside the sky's own photon noise. With three bias frames")
    print("instead of fifteen, the same term would dominate.")